<a href="https://colab.research.google.com/github/UdayKajana/udaykajana-codeforge/blob/artificial_intelligence/playground/dspy_handoff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installations and Imports

In [1]:
!pip install openai-agents dspy
AZURE_API_KEY     = ""
AZURE_ENDPOINT    = ""
AZURE_API_VERSION = ""
AZURE_DEPLOYMENT  = ""

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.0/843.0 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 15.8 MB/s eta 0:00:00
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.5.2
    Uninstalling typeguard-4.5.2:
      Successfully uninstalled typeguard-4.5.2


## Email Extraction

In [4]:
# Importing emails
import json
emails = dict()
with open("/content/bidding_emails.json", encoding="utf-8") as f:
    emails = json.load(f)
for key, body in emails.items():
    print(key, "→", body[:200])


email_01 → FROM: sandeep.rao@delta-infra.com
TO: bids@acmecorp.com
DATE: Monday, May 19, 2026 10:32 AM
SUBJECT: Welding Robots – Looking for a Good Vendor, Thought of You Guys

Hi team,

Hope you're all doing we
email_02 → FROM: supply@greenfield-auto.com
TO: vendors@acmemotors.com
DATE: Tuesday, May 20, 2026 2:15 PM
SUBJECT: RFQ – EV Battery Packs for Q3 Production Run

Dear Vendor,

Good afternoon. We are reaching out
email_03 → FROM: it.buying@novatechsols.com
TO: sales@acmecorp.com
DATE: Wednesday, May 21, 2026 9:48 AM
SUBJECT: Bulk Laptop Procurement – Bid Invitation (Urgent)

Hello,

I hope this finds you well. We are exp
email_04 → FROM: james.whitfield@metro-display.co.uk
TO: procurement.bids@acmecorp.com
DATE: Thursday, May 22, 2026 11:05 AM
SUBJECT: OLED Panel RFQ – 50K Units – Q3/Q4 Delivery

Hi,

We spoke briefly at Display
email_05 → FROM: admin.purchase@globalspace-offices.com
TO: furniture.bids@acmecorp.com
DATE: Friday, May 23, 2026 3:22 PM
SUBJECT: Office Furniture – 2

## Agent-Agent With DSPy - General Usecase

In [ ]:
import dspy
from openai import AsyncAzureOpenAI
from agents import set_default_openai_client, set_default_openai_api
import asyncio
from dataclasses import dataclass
from agents import Agent, Runner, ModelSettings, set_tracing_disabled
set_tracing_disabled(True)


client = AsyncAzureOpenAI(
    api_key        = AZURE_API_KEY,
    azure_endpoint = AZURE_ENDPOINT,
    api_version    = AZURE_API_VERSION,
)
set_default_openai_client(client)
set_default_openai_api("chat_completions")

lm = dspy.LM(
    model       = f"azure/{AZURE_DEPLOYMENT}",
    api_key     = AZURE_API_KEY,
    api_base    = AZURE_ENDPOINT,
    api_version = AZURE_API_VERSION,
    max_tokens  = 1024,
    temperature = 0.0,
)
dspy.configure(lm=lm)

# ── DSPy: raw prompt → JSON skeleton with empty values ───────────────────────

class BuildSkeleton(dspy.Signature):
    """
    Analyze the user request and produce a structured JSON skeleton.

    Rules:
      - Identify the domain and intent of the request first.
      - Derive category names dynamically from the domain — do NOT use hardcoded category names.
      - Keys must be snake_case. Use "" for all values except those explicitly stated in the request.
      - Cover all relevant aspects: who, what, when, where, scope, financials, constraints, context.
      - Group keys into 4–8 logical nested objects suited to the detected domain.
      - Output raw JSON only — no markdown fences, no explanation.
    """
    user_request:  str = dspy.InputField(desc="The raw user request or topic")
    json_skeleton: str = dspy.OutputField(
        desc="Raw JSON with domain-appropriate snake_case keys grouped into logical nested categories. "
             "Pre-fill only values explicitly stated in the request; set all other values to empty string."
    )

def dspy_transform(prompt: str) -> str:
    return dspy.Predict(BuildSkeleton)(user_request=prompt).json_skeleton

# ── SubAgent: receives skeleton → fills every value ──────────────────────────

sub_agent = Agent(
    name="SubAgent",
    model=AZURE_DEPLOYMENT,
    model_settings=ModelSettings(
        max_tokens  = 2048,
        temperature = 0.0,
    ),
    instructions=(
        "You are a data enrichment agent. "
        "You receive a JSON skeleton where most values are empty strings. "
        "Fill in EVERY empty value with accurate, specific, factual information suited to its key and domain. "
        "Preserve the nested structure exactly — do not add, remove, or rename any keys. "
        "Return only the completed JSON — no markdown fences, no explanation."
    ),
)

# ── Pipeline ──────────────────────────────────────────────────────────────────

async def run_pipeline(user_prompt: str) -> str:
    print(f"[User]       {user_prompt}\n")

    # Step 1 — DSPy builds the skeleton
    skeleton = dspy_transform(user_prompt.strip())
    print(f"[DSPy]       {skeleton}\n")

    # Step 2 — SubAgent fills every value
    result = await Runner.run(sub_agent, input=skeleton)
    print(f"[SubAgent]   {result.final_output}")

    return result.final_output

await run_pipeline("Write about china")


## Without DSPy - Email case

In [15]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║         PROCUREMENT EMAIL PIPELINE — No DSPy, Prompt-Config Driven          ║
║                                                                              ║
║  All system prompts and agent settings live in prompts.json.                 ║
║  The pipeline loads them at startup — no hardcoded prompt strings here.      ║
║                                                                              ║
║  prompts.json structure:                                                     ║
║    {                                                                         ║
║      "agent_settings": {                                                     ║
║        "<agent_key>": { name, max_tokens, temperature }                      ║
║      },                                                                      ║
║      "system_prompts": {                                                     ║
║        "<agent_key>": "<full system prompt string>"                          ║
║      }                                                                       ║
║    }                                                                         ║
║                                                                              ║
║  Agent keys (must exist in both sections of prompts.json):                  ║
║    forensic_extractor  → Agent1: reads email, builds structured skeleton     ║
║    structural_auditor  → Agent2: audits structure, quarantines noise         ║
║    enrichment_engine   → Agent3: fills values, resolves, validates           ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import json
import asyncio
from pathlib import Path
from openai import AsyncAzureOpenAI
from agents import (
    Agent, Runner, ModelSettings,
    set_default_openai_client, set_default_openai_api,
    set_tracing_disabled,
)
import nest_asyncio
nest_asyncio.apply()
set_tracing_disabled(True)


# ──────────────────────────────────────────────────────────────────────────────
# Azure credentials
# ──────────────────────────────────────────────────────────────────────────────

azure_client = AsyncAzureOpenAI(
    api_key        = AZURE_API_KEY,
    azure_endpoint = AZURE_ENDPOINT,
    api_version    = AZURE_API_VERSION,
)
set_default_openai_client(azure_client)
set_default_openai_api("chat_completions")


# ──────────────────────────────────────────────────────────────────────────────
# Load prompts config from JSON
# ──────────────────────────────────────────────────────────────────────────────
PROMPTS_FILE = "/content/prompts.json"

def load_prompts(path: Path) -> dict:
    """
    Load and validate prompts.json.
    Returns the full config dict.
    Raises KeyError if any expected agent key is missing from either section.
    """
    with open(path, encoding="utf-8") as f:
        cfg = json.load(f)

    required_keys   = {"forensic_extractor", "structural_auditor", "enrichment_engine"}
    prompts_keys    = set(cfg.get("system_prompts", {}).keys())
    settings_keys   = set(cfg.get("agent_settings", {}).keys())

    missing_prompts  = required_keys - prompts_keys
    missing_settings = required_keys - settings_keys

    if missing_prompts:
        raise KeyError(
            f"prompts.json is missing these keys in 'system_prompts': {missing_prompts}"
        )
    if missing_settings:
        raise KeyError(
            f"prompts.json is missing these keys in 'agent_settings': {missing_settings}"
        )

    return cfg


cfg      = load_prompts(PROMPTS_FILE)
PROMPTS  = cfg["system_prompts"]    # agent_key → prompt string
SETTINGS = cfg["agent_settings"]    # agent_key → { name, max_tokens, temperature }

print(f"[Config] Loaded prompts.json — {len(PROMPTS)} agent prompts registered.")
for key in PROMPTS:
    s = SETTINGS[key]
    print(
        f"  [{key}] name={s['name']} | "
        f"max_tokens={s['max_tokens']} | "
        f"temperature={s['temperature']} | "
        f"prompt_length={len(PROMPTS[key])} chars"
    )


# ──────────────────────────────────────────────────────────────────────────────
# Build agents from config
# Each agent reads its system prompt and settings from the JSON dict by key.
# ──────────────────────────────────────────────────────────────────────────────

def build_agent(key: str) -> Agent:
    """Construct an Agent using settings and prompt from the loaded config."""
    s = SETTINGS[key]
    return Agent(
        name           = s["name"],
        model          = AZURE_DEPLOYMENT,
        model_settings = ModelSettings(
            max_tokens  = s["max_tokens"],
            temperature = s["temperature"],
        ),
        instructions   = PROMPTS[key],   # ← prompt loaded from prompts.json[key]
    )


agent1 = build_agent("forensic_extractor")   # reads email → JSON skeleton
agent2 = build_agent("structural_auditor")   # audits structure → gaps + quarantine
agent3 = build_agent("enrichment_engine")    # fills + resolves + validates → final JSON


# ──────────────────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────────────────

def strip_fences(text: str) -> str:
    """Remove accidental markdown code fences that models sometimes emit."""
    return (
        text.strip()
        .removeprefix("```json")
        .removeprefix("```")
        .removesuffix("```")
        .strip()
    )


# ──────────────────────────────────────────────────────────────────────────────
# Pipeline runner
# ──────────────────────────────────────────────────────────────────────────────

async def run_pipeline(email_key: str, email_text: str) -> dict:
    print(f"\n{'═'*64}")
    print(f"  Processing: {email_key}")
    print(f"{'═'*64}")

    # ── Agent 1 — Forensic Extractor ─────────────────────────────────────────
    # Prompt key: "forensic_extractor"
    # Job: extract only what is stated; build domain-specific tag_list skeleton
    print(f"\n[Agent1 — {SETTINGS['forensic_extractor']['name']}] Reading email...")
    result1       = await Runner.run(agent1, input=email_text)
    agent1_output = result1.final_output
    print(f"[Agent1 done] {len(agent1_output)} chars")

    # ── Agent 2 — Structural Auditor ─────────────────────────────────────────
    # Prompt key: "structural_auditor"
    # Job: add missing fields as ""; quarantine duplicates/irrelevant/ambiguous;
    #      write agent3_briefing in audit_report
    print(f"[Agent2 — {SETTINGS['structural_auditor']['name']}] Auditing structure...")
    result2       = await Runner.run(agent2, input=agent1_output)
    agent2_output = result2.final_output
    print(f"[Agent2 done] {len(agent2_output)} chars")

    # ── Agent 3 — Enrichment Engine ──────────────────────────────────────────
    # Prompt key: "enrichment_engine"
    # Job: fill all "" values; resolve review_section; validate; return final JSON
    print(f"[Agent3 — {SETTINGS['enrichment_engine']['name']}] Enriching and validating...")
    result3      = await Runner.run(agent3, input=agent2_output)
    final_output = result3.final_output
    print(f"[Agent3 done] {len(final_output)} chars")

    # ── Parse final output ────────────────────────────────────────────────────
    try:
        parsed = json.loads(strip_fences(final_output))
    except json.JSONDecodeError as e:
        print(f"[WARNING] JSON parse failed for '{email_key}': {e}")
        parsed = {"raw_output": final_output, "parse_error": str(e)}

    # ── Print summary ─────────────────────────────────────────────────────────
    print(f"\n[FINAL — {email_key}]")
    summary = json.dumps(parsed, indent=2, ensure_ascii=False)
    print(summary)

    return {
        "email_key"     : email_key,
        "agent1_output" : agent1_output,   # raw Agent1 JSON string
        "agent2_output" : agent2_output,   # raw Agent2 JSON string (with audit_report)
        "final"         : parsed,          # parsed final dict
    }


async def run_all(emails: dict) -> list:
    results = []
    for key, body in emails.items():
        result = await run_pipeline(key, body)
        results.append(result)
    return results


# ──────────────────────────────────────────────────────────────────────────────
# Entry point
# ──────────────────────────────────────────────────────────────────────────────

emails = {}
with open("/content/bidding_emails.json", encoding="utf-8") as f:
    emails = json.load(f)

print(f"\nLoaded {len(emails)} emails. Starting pipeline...\n")
results = await run_all(emails)

print(f"\n{'═'*64}")
print(f"  Pipeline complete. {len(results)} emails processed.")
print(f"{'═'*64}")

[Config] Loaded prompts.json — 3 agent prompts registered.
  [forensic_extractor] name=ForensicExtractorAgent | max_tokens=2500 | temperature=0.0 | prompt_length=6964 chars
  [structural_auditor] name=StructuralAuditorAgent | max_tokens=3000 | temperature=0.0 | prompt_length=5880 chars
  [enrichment_engine] name=EnrichmentDecisionAgent | max_tokens=4000 | temperature=0.0 | prompt_length=5992 chars

Loaded 10 emails. Starting pipeline...


════════════════════════════════════════════════════════════════
  Processing: email_01
════════════════════════════════════════════════════════════════

[Agent1 — ForensicExtractorAgent] Reading email...
[Agent1 done] 3293 chars
[Agent2 — StructuralAuditorAgent] Auditing structure...
[Agent2 done] 5976 chars
[Agent3 — EnrichmentDecisionAgent] Enriching and validating...
[Agent3 done] 6894 chars

[FINAL — email_01]
{
  "email_meta": {
    "from_address": "sandeep.rao@delta-infra.com",
    "to_address": "bids@acmecorp.com",
    "date_sent": "Monday, Ma

## With DSPy - Email case

In [ ]:
"""
Pipeline: Agent1 → DSPy → Agent2

Agent1 — reads any document, extracts only what is explicitly stated, inventories all standard fields
DSPy   — structural auditor: adds missing domain-standard fields, flags noise with typed instructions
Agent2 — enrichment + decision engine: fills every gap, resolves flagged items, returns final clean JSON
"""

import json
import asyncio
import dspy
from openai import AsyncAzureOpenAI
from agents import (
    Agent, Runner, ModelSettings,
    set_default_openai_client, set_default_openai_api,
    set_tracing_disabled,
)
import nest_asyncio
nest_asyncio.apply()
set_tracing_disabled(True)


# ─────────────────────────────────────────────────────────────────────────────
# Azure + DSPy Setup
# ─────────────────────────────────────────────────────────────────────────────

azure_client = AsyncAzureOpenAI(
    api_key        = AZURE_API_KEY,
    azure_endpoint = AZURE_ENDPOINT,
    api_version    = AZURE_API_VERSION,
)
set_default_openai_client(azure_client)
set_default_openai_api("chat_completions")

lm = dspy.LM(
    model       = f"azure/{AZURE_DEPLOYMENT}",
    api_key     = AZURE_API_KEY,
    api_base    = AZURE_ENDPOINT,
    api_version = AZURE_API_VERSION,
    max_tokens  = 2048,
    temperature = 0.0,
)
dspy.configure(lm=lm)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Agent 1: Document Extractor
# ─────────────────────────────────────────────────────────────────────────────

AGENT1_INSTRUCTIONS = """
You are a forensic document extractor. Read any incoming document or request and produce a structured JSON.

You operate in TWO modes simultaneously:
  EXTRACT   → Capture every concrete fact the document explicitly states.
  INVENTORY → List every field a complete, standard record of this domain and type requires.

OUTPUT: exactly 5 top-level keys.

────────────────────────────────────────────────────
1. "document_meta"
────────────────────────────────────────────────────
Keys: sender, recipient, date, title_or_subject, document_type
Rule: Fill from document headers. Use null for absent fields.

────────────────────────────────────────────────────
2. "context"
────────────────────────────────────────────────────
Your classification — not the document's words:
  domain       → subject area (e.g. "procurement", "hr", "legal", "finance", "technical", "operations")
  record_type  → document category (e.g. "inquiry", "order", "proposal", "complaint", "contract", "report")
  summary      → 1–2 sentences: what is this document asking for or communicating?
  tone         → one word: "formal" | "informal" | "urgent" | "technical" | "persuasive"
  intent       → "request" | "inform" | "propose" | "complain" | "confirm" | "instruct" | "other"

────────────────────────────────────────────────────
3. "extracted_values"
────────────────────────────────────────────────────
Every concrete fact explicitly stated in the document:
  names, numbers, dates, amounts, specifications, locations, contacts,
  terms, preferences, constraints, references, identifiers.
Rule: Use the document's exact wording and numbers. No paraphrasing. Null if absent.

────────────────────────────────────────────────────
4. "field_inventory"
────────────────────────────────────────────────────
A complete checklist of ALL fields a standard record of this domain/record_type requires.
Construction rules:
  (a) Derive the required field set from domain + record_type — do NOT use hardcoded lists.
  (b) For each required field:
      → Sender stated it: copy the value from extracted_values.
      → Sender did NOT state it: set to "" (empty string, not null).
  (c) All keys: snake_case, lowercase.
  (d) Always cover these universal categories (adapt field names to the domain):
        Identifiers    → reference numbers, IDs, tracking or case numbers
        Subject/Item   → what is being requested, described, or acted upon
        Scope/Quantity → size, volume, count, or extent of the request
        Financial      → amounts, rates, budgets, payment terms, currencies
        Timeline       → start, end, milestones, deadlines, lead times
        Parties        → sender, recipient, approvers, third parties
        Location       → delivery address, service site, jurisdiction
        Requirements   → compliance, standards, certifications, constraints
        Contacts       → names, emails, phone numbers
  (e) Add domain-specific fields based on the detected domain and record_type.
Rule: "" means "field is required but not yet provided." Every standard field must appear.

────────────────────────────────────────────────────
5. "open_items"
────────────────────────────────────────────────────
Array of strings: ambiguities, contradictions, items promised for later, anything a reviewer would flag.
Quote the document where possible. [] if nothing to note.

STRICT RULES:
• Raw JSON only. No markdown fences. No prose before or after.
• Never invent values in extracted_values — only what the document explicitly states.
• All field_inventory keys: snake_case, lowercase.
• Valid, parseable JSON.
"""

agent1 = Agent(
    name           = "DocumentExtractorAgent",
    model          = AZURE_DEPLOYMENT,
    model_settings = ModelSettings(max_tokens=2048, temperature=0.0),
    instructions   = AGENT1_INSTRUCTIONS,
)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — DSPy: Structural Auditor
# ─────────────────────────────────────────────────────────────────────────────

class StructuralAudit(dspy.Signature):
    """
    You are a JSON structure auditor. Your job is structural only — you do NOT fill values.

    Given structured data extracted from a document, you must do three things:
      A. Add standard fields absent from field_inventory (set value to "").
      B. Move problematic fields into review_section with typed resolution instructions.
      C. Write an audit_summary for the next agent covering FILL / RESOLVE / VALIDATE / OUTPUT.

    Never change existing values. Never fill empty strings. Output raw JSON only.
    """
    input_json   : str = dspy.InputField(
        desc="Structured JSON with keys: document_meta, context, extracted_values, field_inventory, open_items"
    )
    audited_json : str = dspy.OutputField(
        desc=(
            "Modified JSON preserving all existing keys, plus: "
            "(1) field_inventory enriched with missing domain-standard fields set to empty string, "
            "(2) problematic fields moved to review_section — each entry has a value and an instruction "
            "typed as one of: DUPLICATE of <key> | IRRELEVANT to <domain> <record_type> | "
            "AMBIGUOUS: suggest <name> | MISPLACED: move to <section>, "
            "(3) audit_summary added with: gaps_added[] (keys added), "
            "quarantined[] (keys moved), completeness_score integer 0-100, "
            "and next_agent_briefing string covering FILL / RESOLVE / VALIDATE / OUTPUT sub-sections."
        )
    )


def dspy_audit(agent1_output: str) -> str:
    return dspy.Predict(StructuralAudit)(input_json=agent1_output).audited_json


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Agent 2: Enrichment & Decision Engine
# ─────────────────────────────────────────────────────────────────────────────

AGENT2_INSTRUCTIONS = """
You are a senior data enrichment and decision agent. You receive a JSON that has been extracted from
a document and structurally audited. Complete it, resolve all open questions, and return the final clean record.

READ FIRST:
1. Read "audit_summary.next_agent_briefing" — this is your instruction set.
2. Note "audit_summary.gaps_added" — newly added fields that all need filling.
3. Note "audit_summary.quarantined" — fields in review_section awaiting decisions.
4. Do NOT change document_meta, context, or extracted_values — they are finalized.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TASK 1 — FILL field_inventory (highest priority)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
For every field where value is "" (empty string):

  STEP 1 — DERIVE: Can you compute it from other filled fields?
           (e.g. total_value = quantity × unit_price, end_date = start_date + duration)
           Use arithmetic or date math where applicable.

  STEP 2 — DOMAIN KNOWLEDGE: What is the standard or typical value for this field in this domain?
           Provide specific values, not vague placeholders. State the basis where helpful
           (e.g. "Net 30 (industry standard)", "ISO 9001 (standard for manufacturing)").

  STEP 3 — LAST RESORT: If genuinely unknowable without more input, set to "__NEEDS_INPUT__".

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TASK 2 — RESOLVE review_section
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
For each item, read its instruction and execute:
  DUPLICATE  → merge value into canonical field; drop duplicate
  IRRELEVANT → drop it (keep only if clearly applicable despite the flag)
  AMBIGUOUS  → apply suggested rename and fill; drop if unresolvable
  MISPLACED  → move to the correct section

Record every decision in a "resolution_log":
  { "<key>": { "action": "merged|dropped|renamed|moved", "destination": "...", "reason": "..." } }

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TASK 3 — VALIDATE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Add a "validation" section with:
  "completeness"    → "complete" | "mostly_complete" | "incomplete"
  "missing_inputs"  → array of field_inventory keys still set to "__NEEDS_INPUT__"
  "inconsistencies" → array of contradictions or mismatches found between fields
  "risk_flags"      → array of concerns a reviewer would escalate
  "summary"         → one sentence assessing overall record quality and readiness

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT RULES (non-negotiable):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Return exactly these top-level keys:
    document_meta, context, extracted_values, field_inventory, open_items, resolution_log, validation
• REMOVE: review_section, audit_summary
• Every field_inventory value must be filled or "__NEEDS_INPUT__" — no "" remaining
• Raw JSON only. No markdown fences. Valid JSON.
"""

agent2 = Agent(
    name           = "DataEnrichmentAgent",
    model          = AZURE_DEPLOYMENT,
    model_settings = ModelSettings(max_tokens=4000, temperature=0.0),
    instructions   = AGENT2_INSTRUCTIONS,
)


# ─────────────────────────────────────────────────────────────────────────────
# Pipeline runner
# ─────────────────────────────────────────────────────────────────────────────

async def run_pipeline(email_key: str, email_text: str) -> dict:
    print(f"\n{'='*60}")
    print(f"Processing: {email_key}")
    print(f"{'='*60}")

    # ── Step 1: Agent1 extracts context + builds field inventory ──────────────
    print(f"\n[Agent1] Extracting and classifying document...")
    result1 = await Runner.run(agent1, input=email_text)
    agent1_output = result1.final_output
    print(f"[Agent1 done] {len(agent1_output)} chars")

    # ── Step 2: DSPy audits structure, fills gaps, quarantines noise ──────────
    print(f"[DSPy] Running structural audit...")
    dspy_output = dspy_audit(agent1_output)
    print(f"[DSPy done] {len(dspy_output)} chars")

    # ── Step 3: Agent2 enriches, decides, and validates ───────────────────────
    print(f"[Agent2] Enriching, resolving, and validating...")
    result2 = await Runner.run(agent2, input=dspy_output)
    final_output = result2.final_output
    print(f"[Agent2 done] {len(final_output)} chars")

    # ── Parse + strip any accidental markdown fences ──────────────────────────
    try:
        clean = (
            final_output.strip()
            .removeprefix("```json")
            .removeprefix("```")
            .removesuffix("```")
            .strip()
        )
        parsed = json.loads(clean)
    except json.JSONDecodeError as e:
        print(f"[WARNING] JSON parse failed for {email_key}: {e}")
        parsed = {"raw_output": final_output, "parse_error": str(e)}

    return {
        "email_key"    : email_key,
        "agent1_output": agent1_output,
        "dspy_output"  : dspy_output,
        "final"        : parsed,
    }


async def run_all(emails: dict) -> list:
    results = []
    for key, body in emails.items():
        result = await run_pipeline(key, body)
        results.append(result)
        print(f"\n[FINAL — {key}]")
        print(json.dumps(result["final"], indent=2, ensure_ascii=False), "\n")
    return results


# ─────────────────────────────────────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────────────────────────────────────

import json
emails = dict()
with open("/content/bidding_emails.json", encoding="utf-8") as f:
    emails = json.load(f)
print(f"\nLoaded {len(emails)} emails. Starting pipeline...\n")
results = await run_all(emails)
print(f"\n{'='*60}")
print(f"Pipeline complete. {len(results)} emails processed.")
print(f"{'='*60}")
